<a href="https://colab.research.google.com/github/kiranUlhasKulkarni/JobValidator/blob/main/JobValidator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI-Powered Resume & Job Description Validator
**Author:** Kiran Kulkarni

### Overview
This tool leverages Google's Gemini API to evaluate a resume against a specific Job Description. It utilizes a custom-built dynamic fallback router to ensure high availability and bypass server bottlenecks.

**Key Features:**
* **Dynamic Failover Routing:** Automatically catches 503/404 errors and iterates through available models to ensure the evaluation completes.
* **Two-Phase Architecture:**
  1. Calculates a strict numerical match percentage.
  2. If the score falls below the set threshold, a secondary autonomous agent triggers to perform a targeted gap analysis.
* **Secure Environment:** Utilizes Google Colab Secrets for API key management.

In [ ]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 707.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 790.4/790.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.6/240.6 kB 15.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.2 which is incompatible.


In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('geminiKey')
print("API Key secured and loaded.")

API Key secured and loaded.


In [ ]:
my_resume = "job_description = "threshold = 50 # @param {type:"slider", min:0, max:100, step:5}

print("Inputs captured. Ready to evaluate.")

Inputs captured. Ready to evaluate.


In [ ]:
import requests
import os
from google.colab import userdata

# 1. Secure the Key
try:
    API_KEY = userdata.get('geminiKey')
except:
    print("Error: Could not find 'geminiKey' in Colab Secrets.")

# --- THE NEW DYNAMIC ROUTER METHOD ---
def execute_with_failover(prompt_text, available_models):
    """Tries every model in the list until one succeeds."""
    payload = {"contents": [{"parts": [{"text": prompt_text}]}], "generationConfig": {"temperature": 0}}

    for model_name in available_models:
        URL = f"https://generativelanguage.googleapis.com/v1beta/{model_name}:generateContent?key={API_KEY}"
        try:
            raw_response = requests.post(URL, json=payload).json()

            # Catch 503s or 404s and loop to the next!
            if 'error' in raw_response:
                print(f"  [!] {model_name} busy. Initiating failover...")
                continue

            # If it works, extract text and return it along with the model used
            text = raw_response['candidates'][0]['content']['parts'][0]['text'].strip()
            print(f"  [+] Success: Data retrieved via {model_name}")
            return text, model_name

        except Exception as e:
            continue

    return None, None # Returns None if absolutely every server is down

def run_evaluation():
    if not my_resume or not job_description:
        print("Error: Please fill in the Resume and JD fields.")
        return

    print("System Booting: Fetching live server topology...")

    # 1. Get the list of allowed models ONCE at boot
    models_url = f"https://generativelanguage.googleapis.com/v1beta/models?key={API_KEY}"
    models_response = requests.get(models_url).json()

    if 'error' in models_response:
        print(f"\nAPI Key Error: {models_response['error']['message']}")
        return

    allowed_models = [m['name'] for m in models_response.get('models', []) if 'generateContent' in m.get('supportedGenerationMethods', [])]
    sorted_models = sorted(allowed_models, key=lambda x: 'flash' not in x.lower())

    if not sorted_models:
        print("Fatal Error: No text models available.")
        return

    # ---------------------------------------------------------
    # PHASE 1: SCORING (Uses dynamic router)
    # ---------------------------------------------------------
    print("\nExecuting Phase 1: Match Evaluation...")
    prompt_1 = f"Compare this Resume to the Job Description. Output ONLY a single numerical score between 0 and 100.\n\nResume: {my_resume}\nJD: {job_description}"

    score_text, _ = execute_with_failover(prompt_1, sorted_models)

    if not score_text:
        print("\nCRITICAL FAILURE: All models are currently down.")
        return

    try:
        score = int(''.join(filter(str.isdigit, score_text)))
    except ValueError:
        score = 50 # Fallback if parsing fails

    print(f"\n{'='*30}\nMATCH SCORE: {score}%\n{'='*30}")

    if score >= threshold:
        print("RESULT: PASS - Candidate meets requirements.")
    else:
        print("RESULT: FAIL (Below Threshold)")

        # ---------------------------------------------------------
        # PHASE 2: GAP ANALYSIS (Uses dynamic router AGAIN independently)
        # ---------------------------------------------------------
        print("\nExecuting Phase 2: Gap Analysis...")
        prompt_2 = f"The candidate scored {score}%. Identify 3 specific technical gaps they need to bridge for this role. Suggest one Northeastern University course or project to help. \n\nResume: {my_resume}\nJD: {job_description}"

        gap_text, _ = execute_with_failover(prompt_2, sorted_models)

        if not gap_text:
             print("\n[!] Gap Analysis API Error: All servers are currently under extreme load. Try again later.")
        else:
             gap_text = gap_text.replace("*", "")
             print("\n--- GAP ANALYSIS ---")
             print(gap_text)

run_evaluation()

System Booting: Fetching live server topology...

Executing Phase 1: Match Evaluation...
  [+] Success: Data retrieved via models/gemini-2.5-flash

MATCH SCORE: 97%
RESULT: PASS - Candidate meets requirements.
